# Minilab EduBI — Eksplorasi Data

Notebook ini digunakan untuk eksplorasi interaktif data dari **ClickHouse Data Warehouse**.

**Prasyarat**: Sudah jalankan ETL dan dbt:
```bash
docker compose run --rm app
docker compose run --rm dbt
```

**Install library** (jika belum):
```bash
pip install clickhouse-connect pandas
```

In [ ]:
import clickhouse_connect
import pandas as pd

# Koneksi ke ClickHouse (harus jalan di localhost:8123)
client = clickhouse_connect.get_client(
    host='localhost',
    port=8123,
    username='default',
    password=''
)
print('Terhubung ke ClickHouse.')

In [ ]:
# Lihat semua database
pd.DataFrame(client.query('SHOW DATABASES').result_rows, columns=['database'])

## Bronze Layer

In [ ]:
# Lihat tabel di database bronze
pd.DataFrame(client.query('SHOW TABLES FROM bronze').result_rows, columns=['table'])

In [ ]:
# Preview bronze.sales
client.query_df('SELECT * FROM bronze.sales LIMIT 5')

In [ ]:
# Jumlah baris per tabel bronze
for tbl in ['sales', 'customers', 'reviews', 'targets']:
    n = client.query(f'SELECT count() FROM bronze.{tbl}').result_rows[0][0]
    print(f'bronze.{tbl}: {n} baris')

## Silver Layer

In [ ]:
# Preview silver.silver_sales
client.query_df('SELECT * FROM silver.silver_sales LIMIT 5')

In [ ]:
# Distribusi revenue_category
client.query_df('''
    SELECT revenue_category, count() AS jumlah
    FROM silver.silver_sales
    GROUP BY revenue_category
    ORDER BY jumlah DESC
''')

In [ ]:
# Distribusi sentimen ulasan
client.query_df('''
    SELECT sentiment, count() AS jumlah
    FROM silver.silver_reviews
    GROUP BY sentiment
    ORDER BY jumlah DESC
''')

## Gold Layer

In [ ]:
# KPI per cabang
client.query_df('''
    SELECT
        branch,
        total_orders,
        total_revenue,
        avg_order_value,
        revenue_achievement_pct,
        avg_rating
    FROM gold.gold_branch_kpi
    ORDER BY total_revenue DESC
''')

In [ ]:
# Tren revenue per bulan
client.query_df('''
    SELECT
        order_year,
        order_month,
        sum(total_revenue) AS monthly_revenue,
        sum(total_orders)  AS monthly_orders
    FROM gold.gold_sales_daily
    GROUP BY order_year, order_month
    ORDER BY order_year, order_month
''')

In [ ]:
# Ringkasan ulasan per cabang
client.query_df('''
    SELECT branch, avg_rating, total_reviews, pct_positif
    FROM gold.gold_review_summary
    ORDER BY avg_rating DESC
''')